# Cav1.2 MolProbity / clashscore analysis

Vanilla WT/G402S/G406R and the available corrected WT masked candidate are shown. Corrected masked mutant cohorts are not yet available; retired masked mutant CSVs are excluded.
The nearby short distances in `Cav12_G406R_mutationSite_analysis.ipynb` motivate the question; they
do not establish that any global clashscore change originates at introduced R406.

## Reproducible setup
Run from the repository root or channel directory. Dependencies: NumPy, pandas, SciPy, statsmodels, Matplotlib, seaborn, and Jupyter. MolProbity tables remain in the local downloaded directory. No upload or move is performed. Shared functions live in `shared/molprobity_analysis.py`, as in the existing analyses.

In [1]:
from pathlib import Path
import sys, os, json, platform, importlib.metadata
from IPython.display import display, Markdown
import matplotlib.pyplot as plt
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'shared/data_access.py').is_file())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from shared import molprobity_analysis as mp
CHANNEL = 'cav12'
OUT = mp.output_dir(CHANNEL)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 24)
mp.apply_kv21_style()


## Data-access boundary
Set `VGIC_MOLPROBITY_ROOT` or `DATA_ROOT` to the downloaded directory. Future Hugging Face migration only needs to replace the loader with explicit paths through the existing `shared.data_access.resolve`; downstream analysis accepts the same DataFrame. Existing structural-QC and conformational inputs already use that resolver, including local/cache/offline behavior.

In [2]:
DATA_ROOT = os.environ.get('VGIC_MOLPROBITY_ROOT')
raw, inventory, total_csv_files = mp.load_molprobity(CHANNEL, DATA_ROOT)
inventory.to_csv(OUT/'tables/input_inventory.csv', index=False)
print(f'{total_csv_files} CSVs in downloaded directory; {len(inventory)} for this channel; {len(raw):,} rows')
display(inventory)


12 CSVs in downloaded directory; 6 for this channel; 36,000 rows


,csv_file,rows,sha256
0,Cav12_g402s_masked_v3_50pct_clashscores.csv,6000,33947bad81d52c08ffec2cf26bc9c41a075799b0ec47c9...
1,Cav12_g402s_vanilla_clashscores.csv,6000,d09dad6952d455bcb6b67d2669ceb7c4c3bbf46dbcd055...
2,Cav12_g406r_masked_v3_50pct_clashscores.csv,6000,fbaee13984c1ed4b56f1717cddc80308e7ff4cde4ef46a...
3,Cav12_g406r_vanilla_clashscores.csv,6000,7698d8f7b8967fad29bcaa50ead4b292047e160c141f37...
4,Cav12_wt_masked_v3_50pct_clashscores.csv,6000,25b0fcda0d6f250ae5ab74837d1b13cc313d5c930f502c...
5,Cav12_wt_vanilla_clashscores.csv,6000,ad9855b8606eb332e9dc22c0ab41acd9c99a3651edfa75...


## Audit before plotting
The expected grid uses each ensemble’s recorded seed labels, AF models 1–5, r0–r10, and final. Missing whole trajectories/seeds are additionally checked against the existing model manifest. Raw input hashes are saved above. NaN recycle is expected only for final/base models; missing rank is allowed and reported.

In [3]:
data, audit = mp.audit_data(raw, CHANNEL, OUT)
display(audit)
display(pd.read_csv(OUT/'tables/missing_snapshots.csv'))
display(pd.read_csv(OUT/'tables/manifest_discrepancies.csv'))
display(pd.read_csv(OUT/'tables/row_anomalies.csv'))
cohorts = mp.select_cohorts(data, CHANNEL, OUT)
summary = pd.concat([mp.describe(frame, name) for name, frame in cohorts.items()], ignore_index=True)
summary.to_csv(OUT/'tables/descriptive_statistics.csv', index=False)
display(summary[(summary.cohort=='all_recorded') & summary.metric.eq('clashscore')].round(3))
display(pd.read_csv(OUT/'tables/selection_audit.csv'))


Audit: 36,000 rows, 0 failed, 0 invalid metric rows; 0 absent expected snapshots; 0 explicitly excluded rows.


,ensemble,rows,ok,ok_fraction,eligible,failed,missing_clashscore,nonfinite_clashscore,duplicate_names,duplicate_keys,missing_grid_snapshots,seeds,seed_min,seed_max,missing_rank,final_r10_pairs,final_r10_identical_metrics,manifest_rows,manifest_missing,manifest_extra
0,G402S | masked v3,6000,6000,1.0,6000,0,0,0,0,0,0,100,33,132,0,500,500,6000,0,0
1,G402S | vanilla,6000,6000,1.0,6000,0,0,0,0,0,0,100,91,190,0,500,500,6000,0,0
2,G406R | masked v3,6000,6000,1.0,6000,0,0,0,0,0,0,100,33,132,0,500,500,6000,0,0
3,G406R | vanilla,6000,6000,1.0,6000,0,0,0,0,0,0,100,91,190,0,500,500,6000,0,0
4,WT | masked v3,6000,6000,1.0,6000,0,0,0,0,0,0,100,33,132,0,500,500,6000,0,0
5,WT | vanilla,6000,6000,1.0,6000,0,0,0,0,0,0,100,91,190,0,500,500,6000,0,0


,ensemble,seed,af_model,snapshot


,ensemble,model_name,issue


,run_name,input_folder,model_name,rank,af_model,seed,recycle,snapshot,clashscore,n_bad_clashes,max_overlap,status,...,source_path,genotype,protocol,ensemble,csv_file,analysis_eligible,malformed_numeric,invalid_identity,duplicate_name,duplicate_key,invalid_metric,failed_status_or_exit


,cohort,ensemble,metric,N,n_seeds,median,Q1,Q3,IQR,mean,SD,minimum,maximum
0,all_recorded,G402S | masked v3,clashscore,6000,100,33.130,30.540,39.132,8.593,35.339,6.836,26.52,78.80
3,all_recorded,G402S | vanilla,clashscore,6000,100,30.590,28.920,37.942,9.022,33.317,6.583,24.97,68.54
6,all_recorded,G406R | masked v3,clashscore,6000,100,32.790,30.600,38.830,8.230,35.160,6.849,26.10,74.45
9,all_recorded,G406R | vanilla,clashscore,6000,100,31.160,29.310,37.918,8.607,33.611,6.384,24.96,67.17
12,all_recorded,WT | masked v3,clashscore,6000,100,32.685,30.398,38.670,8.272,34.972,6.831,25.41,76.80
15,all_recorded,WT | vanilla,clashscore,6000,100,30.625,29.040,37.782,8.742,33.371,6.692,23.79,70.57


,ensemble,selection,source,requested,matched,excluded_failed,excluded_by_structural_qc
0,G402S | masked v3,3 Å convergence QC,cav12/rmsd_convergence_filtering/Cav12_rmsd_co...,4993,4993,0,1007
1,G402S | vanilla,3 Å convergence QC,cav12/rmsd_convergence_filtering/Cav12_rmsd_co...,5000,5000,0,1000
2,G406R | masked v3,3 Å convergence QC,cav12/rmsd_convergence_filtering/Cav12_rmsd_co...,4998,4998,0,1002
3,G406R | vanilla,3 Å convergence QC,cav12/rmsd_convergence_filtering/Cav12_rmsd_co...,5000,5000,0,1000
4,WT | masked v3,3 Å convergence QC,cav12/rmsd_convergence_filtering/Cav12_rmsd_co...,4992,4992,0,1008
5,WT | vanilla,3 Å convergence QC,cav12/rmsd_convergence_filtering/Cav12_rmsd_co...,5000,5000,0,1000


## Statistical design and interpretation

The complete downloaded table is audited before filtering. `status != ok`, nonzero exit codes,
nonfinite/negative diagnostics and malformed identities are reported explicitly. Ambiguous identities
or mismatched manifests stop execution. Empty anomaly CSVs still retain their headers.

Every ensemble has up to 100 seeds × 5 AF models × 11 recycle snapshots plus a final/base file.
The final/base file has identical recorded MolProbity metrics to r10 for every available trajectory;
that is **metric equality, not a coordinate checksum**. To avoid double weighting those observations,
`all_recycles` excludes final copies. `final_only` is a separate sensitivity analysis.

The primary scientific comparison uses the repository's existing QC subset, independently of clashscore:
3 Å convergence for Cav1.2/Nav1.5 and the established structural/interface/alignment-QC distance
selection for Kv2.1. All raw distributions and excluded counts remain visible. QC-selected results
describe survivors, not all generated models. No new clashscore cutoff is used.

For each metric, average snapshots within seed/AF-model trajectory, then average AF models within seed.
This gives each retained seed equal weight and avoids treating recycles or the five AF models as
independent replicates. Point estimates are seed-balanced mean differences B−A; 95% percentile
bootstrap intervals use 3,000 seed draws. When seed-label sets match completely, nominal paired-seed
t tests and the existing `paired_common_seed_bootstrap` are used. Otherwise Welch tests compare all
retained seed summaries. Both independent-seed and common-label paired sensitivities are exported.
Shared numeric labels do **not** establish shared random-number streams; no seeds are renumbered,
and rank is never a pairing key. Common-label sensitivity may change the population when ranges differ.

BH correction covers all planned pairwise comparisons × all three metrics within each channel/cohort;
primary, independent, and common-label p-values form separate families. Supplemental cohort families
are exploratory. The seed rank-biserial effect is P(B>A)−P(B<A) across seed summaries, with ties zero;
it describes marginal ordering even for nominally paired tests. Model medians/IQRs are descriptive
snapshot summaries and are a different estimand from the seed-balanced mean contrast.

AF-model/snapshot-adjusted regressions provide a sensitivity check with standard errors clustered by
recorded seed. Rank is an output ordering variable, is absent for Kv2.1 L403A vanilla, and is inspected
descriptively rather than controlled as if it were a randomized covariate. These uncertainty estimates
describe computational sampling, not biological replication or a validated clinical effect.

Clashscore is a normalized global clash statistic ([Phenix glossary](https://www.phenix-online.org/version_docs/1.8.2-1296/glossary.htm)).
`n_bad_clashes` is an absolute count and `max_overlap` is a single extreme; they are complementary
diagnostics, not interchangeable biological measures. The local export does not record Phenix version,
hydrogen options, the atom denominator, or atom-level clash identities. Absolute validation grades,
cross-channel biological comparisons, and residue-specific attribution require additional provenance.


In [4]:
contrasts = mp.contrast_statistics(cohorts, CHANNEL, OUT)
primary = contrasts[(contrasts.cohort=='repository_QC') & contrasts.metric.eq('clashscore')]
display(primary[['A','B','N_A','N_B','n_seeds_A','n_seeds_B','common_seeds',
                 'common_model_snapshots','only_A_snapshots','only_B_snapshots',
                 'paired_seed_correlation','primary_test']])
display(primary[['A','B','median_difference','seed_mean_difference','CI_low','CI_high',
                 'seed_rank_biserial','q_BH','independent_q_BH','common_seed_difference',
                 'common_seed_CI_low','common_seed_CI_high','common_seed_q_BH']].round(4))


,A,B,N_A,N_B,n_seeds_A,n_seeds_B,common_seeds,common_model_snapshots,only_A_snapshots,only_B_snapshots,paired_seed_correlation,primary_test
27,WT | vanilla,WT | masked v3,5000,4992,100,100,42,2096,2904,2896,0.293168,Welch seed t
30,G402S | vanilla,G402S | masked v3,5000,4993,100,100,42,2095,2905,2898,0.057503,Welch seed t
33,G406R | vanilla,G406R | masked v3,5000,4998,100,100,42,2099,2901,2899,0.189686,Welch seed t
36,WT | vanilla,G402S | vanilla,5000,5000,100,100,100,5000,0,0,-0.050805,paired nominal seed t
39,WT | vanilla,G406R | vanilla,5000,5000,100,100,100,5000,0,0,0.090012,paired nominal seed t
42,G402S | vanilla,G406R | vanilla,5000,5000,100,100,100,5000,0,0,-0.131691,paired nominal seed t
45,WT | masked v3,G402S | masked v3,4992,4993,100,100,100,4985,7,8,-0.009239,paired nominal seed t
48,WT | masked v3,G406R | masked v3,4992,4998,100,100,100,4990,2,8,-0.048750,paired nominal seed t
51,G402S | masked v3,G406R | masked v3,4993,4998,100,100,100,4991,2,7,0.079072,paired nominal seed t


,A,B,median_difference,seed_mean_difference,CI_low,CI_high,seed_rank_biserial,q_BH,independent_q_BH,common_seed_difference,common_seed_CI_low,common_seed_CI_high,common_seed_q_BH
27,WT | vanilla,WT | masked v3,1.950,1.5232,1.4436,1.5985,1.0000,0.0000,0.000,1.5107,1.4094,1.6126,0.0000
30,G402S | vanilla,G402S | masked v3,2.250,1.9707,1.8910,2.0485,1.0000,0.0000,0.000,1.9571,1.8423,2.0680,0.0000
33,G406R | vanilla,G406R | masked v3,1.545,1.4487,1.3801,1.5180,1.0000,0.0000,0.000,1.4746,1.3832,1.5605,0.0000
36,WT | vanilla,G402S | vanilla,-0.040,-0.0690,-0.1489,0.0101,-0.1156,0.1034,0.093,-0.0690,-0.1489,0.0101,0.1034
39,WT | vanilla,G406R | vanilla,0.505,0.2587,0.1899,0.3252,0.5095,0.0000,0.000,0.2587,0.1899,0.3252,0.0000
42,G402S | vanilla,G406R | vanilla,0.545,0.3277,0.2514,0.4016,0.6346,0.0000,0.000,0.3277,0.2514,0.4016,0.0000
45,WT | masked v3,G402S | masked v3,0.260,0.3785,0.3012,0.4547,0.6426,0.0000,0.000,0.3785,0.3012,0.4547,0.0000
48,WT | masked v3,G406R | masked v3,0.100,0.1842,0.1081,0.2546,0.3812,0.0000,0.000,0.1842,0.1081,0.2546,0.0000
51,G402S | masked v3,G406R | masked v3,-0.160,-0.1943,-0.2694,-0.1254,-0.3985,0.0000,0.000,-0.1943,-0.2694,-0.1254,0.0000


## Clashscore distributions and effect sizes
Violins retain every eligible individual snapshot with median and quartiles; no outliers are clipped. The paired raw/QC panels expose selection effects. The contrast figure shows seed-balanced differences and 95% bootstrap intervals; q-values are reported without significance stars. Colors, typography, thin outlines, light grids, and PDF/300-dpi PNG exports follow `shared.plotting` and existing conformational figures.

In [5]:
mp.plot_distributions(cohorts, CHANNEL, OUT)
plt.show()
mp.plot_effects(contrasts, CHANNEL, OUT)
plt.show()
display(Markdown(mp.interpretation(summary, contrasts, CHANNEL)))


/var/folders/m9/fym_fwtx3554t2jjfnw3s5cw0000gp/T/ipykernel_78632/2804364640.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/m9/fym_fwtx3554t2jjfnw3s5cw0000gp/T/ipykernel_78632/2804364640.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Results on the repository QC subset

Model medians describe retained snapshots. Differences and confidence intervals below use equal AF-model weights within seed and seed-level inference.
- **WT | masked v3 versus WT | vanilla:** model medians 32.35 versus 30.40 (Δ +1.95); seed-balanced mean Δ +1.52, 95% CI [1.44, 1.60]; seed rank-biserial effect +1.00; BH q=3.16e-93.
- **G402S | masked v3 versus G402S | vanilla:** model medians 32.61 versus 30.36 (Δ +2.25); seed-balanced mean Δ +1.97, 95% CI [1.89, 2.05]; seed rank-biserial effect +1.00; BH q=2.24e-111.
- **G406R | masked v3 versus G406R | vanilla:** model medians 32.45 versus 30.91 (Δ +1.55); seed-balanced mean Δ +1.45, 95% CI [1.38, 1.52]; seed rank-biserial effect +1.00; BH q=2.13e-98.
- **G402S | vanilla versus WT | vanilla:** model medians 30.36 versus 30.40 (Δ -0.04); seed-balanced mean Δ -0.07, 95% CI [-0.15, 0.01]; seed rank-biserial effect -0.12; BH q=0.103.
- **G406R | vanilla versus WT | vanilla:** model medians 30.91 versus 30.40 (Δ +0.51); seed-balanced mean Δ +0.26, 95% CI [0.19, 0.33]; seed rank-biserial effect +0.51; BH q=2.25e-10.
- **G406R | vanilla versus G402S | vanilla:** model medians 30.91 versus 30.36 (Δ +0.55); seed-balanced mean Δ +0.33, 95% CI [0.25, 0.40]; seed rank-biserial effect +0.63; BH q=4.12e-13.
- **G402S | masked v3 versus WT | masked v3:** model medians 32.61 versus 32.35 (Δ +0.26); seed-balanced mean Δ +0.38, 95% CI [0.30, 0.45]; seed rank-biserial effect +0.64; BH q=4.02e-15.
- **G406R | masked v3 versus WT | masked v3:** model medians 32.45 versus 32.35 (Δ +0.10); seed-balanced mean Δ +0.18, 95% CI [0.11, 0.25]; seed rank-biserial effect +0.38; BH q=6.04e-06.
- **G406R | masked v3 versus G402S | masked v3:** model medians 32.45 versus 32.61 (Δ -0.16); seed-balanced mean Δ -0.19, 95% CI [-0.27, -0.13]; seed rank-biserial effect -0.40; BH q=1.68e-06.

## Complementary steric diagnostics
These panels use the repository QC subset. Counts depend on structure size and atom handling; maximum overlap reports one extreme contact. Full numerical results and corrected comparisons for both metrics are retained alongside clashscore.

In [6]:
mp.plot_diagnostics(cohorts, CHANNEL, OUT)
plt.show()
display(summary[(summary.cohort=='repository_QC') & summary.metric.ne('clashscore')].round(3))
display(contrasts[(contrasts.cohort=='repository_QC') & contrasts.metric.ne('clashscore')][
    ['A','B','metric','median_difference','seed_mean_difference','CI_low','CI_high','seed_rank_biserial','q_BH']].round(4))


/var/folders/m9/fym_fwtx3554t2jjfnw3s5cw0000gp/T/ipykernel_78632/1691443703.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,cohort,ensemble,metric,N,n_seeds,median,Q1,Q3,IQR,mean,SD,minimum,maximum
55,repository_QC,G402S | masked v3,n_bad_clashes,4993,100,883.000,825.000,995.000,170.000,930.701,144.263,718.000,1586.000
56,repository_QC,G402S | masked v3,max_overlap,4993,100,1.625,1.594,1.653,0.059,1.621,0.048,1.400,1.785
58,repository_QC,G402S | vanilla,n_bad_clashes,5000,100,822.000,781.000,908.000,127.000,877.419,146.903,676.000,1446.000
59,repository_QC,G402S | vanilla,max_overlap,5000,100,1.607,1.567,1.640,0.073,1.599,0.062,1.342,1.772
61,repository_QC,G406R | masked v3,n_bad_clashes,4998,100,879.000,826.000,981.000,155.000,925.809,143.242,707.000,1566.000
62,repository_QC,G406R | masked v3,max_overlap,4998,100,1.623,1.590,1.651,0.061,1.619,0.049,1.384,1.762
64,repository_QC,G406R | vanilla,n_bad_clashes,5000,100,837.000,792.000,914.000,122.000,886.695,141.909,676.000,1496.000
65,repository_QC,G406R | vanilla,max_overlap,5000,100,1.607,1.567,1.640,0.073,1.601,0.056,1.324,1.752
67,repository_QC,WT | masked v3,n_bad_clashes,4992,100,876.000,820.000,977.000,157.000,920.107,142.326,688.000,1989.000
68,repository_QC,WT | masked v3,max_overlap,4992,100,1.620,1.590,1.647,0.057,1.617,0.046,1.397,1.763


,A,B,metric,median_difference,seed_mean_difference,CI_low,CI_high,seed_rank_biserial,q_BH
28,WT | vanilla,WT | masked v3,n_bad_clashes,53.000,41.2927,39.1434,43.3299,1.0000,0.0000
29,WT | vanilla,WT | masked v3,max_overlap,0.015,0.0197,0.0176,0.0217,0.9492,0.0000
31,G402S | vanilla,G402S | masked v3,n_bad_clashes,61.000,53.4027,51.2450,55.5018,1.0000,0.0000
32,G402S | vanilla,G402S | masked v3,max_overlap,0.018,0.0224,0.0203,0.0245,0.9651,0.0000
34,G406R | vanilla,G406R | masked v3,n_bad_clashes,42.000,39.2143,37.3488,41.0976,1.0000,0.0000
35,G406R | vanilla,G406R | masked v3,max_overlap,0.016,0.0177,0.0154,0.0198,0.8857,0.0000
37,WT | vanilla,G402S | vanilla,n_bad_clashes,-1.000,-1.7442,-3.8970,0.3966,-0.1056,0.1199
38,WT | vanilla,G402S | vanilla,max_overlap,0.002,0.0013,-0.0009,0.0033,0.0655,0.2516
40,WT | vanilla,G406R | vanilla,n_bad_clashes,14.000,7.5316,5.6677,9.3318,0.5432,0.0000
41,WT | vanilla,G406R | vanilla,max_overlap,0.002,0.0037,0.0015,0.0059,0.2577,0.0015


## Recycle, AF-model, seed, and rank sensitivity
The full snapshot trace includes final separately and all early-recycle tails. The final-only analysis does not substitute a last available snapshot for a missing final file. The adjusted regression controls AF-model and snapshot composition, with uncertainty clustered by seed; its observation-weighted estimand differs from the seed-balanced primary result.

In [7]:
mp.stratification(cohorts, CHANNEL, OUT)
plt.show()
adjusted = mp.adjusted_contrasts(cohorts, CHANNEL, OUT)
display(adjusted.round(4))
af_sensitivity = mp.af_model_sensitivity(cohorts, CHANNEL, OUT)
display(af_sensitivity.pivot(index=['A','B','mode'], columns='AF_model', values='seed_mean_difference').round(3))
display(contrasts[contrasts.metric.eq('clashscore')][
    ['cohort','A','B','seed_mean_difference','CI_low','CI_high','q_BH']].round(3))


/var/folders/m9/fym_fwtx3554t2jjfnw3s5cw0000gp/T/ipykernel_78632/2647556288.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,A,B,adjusted_difference,CI_low,CI_high,p,q_BH
0,WT | vanilla,WT | masked v3,1.5324,1.4616,1.6033,0.000,0.000
1,G402S | vanilla,G402S | masked v3,1.9779,1.9014,2.0545,0.000,0.000
2,G406R | vanilla,G406R | masked v3,1.4504,1.3843,1.5165,0.000,0.000
3,WT | vanilla,G402S | vanilla,-0.0690,-0.1486,0.0105,0.089,0.089
4,WT | vanilla,G406R | vanilla,0.2587,0.1881,0.3293,0.000,0.000
5,G402S | vanilla,G406R | vanilla,0.3277,0.2521,0.4033,0.000,0.000
6,WT | masked v3,G402S | masked v3,0.3767,0.2987,0.4546,0.000,0.000
7,WT | masked v3,G406R | masked v3,0.1766,0.1036,0.2497,0.000,0.000
8,G402S | masked v3,G406R | masked v3,-0.2001,-0.2727,-0.1275,0.000,0.000


AF_model                                                    1      2      3  \
A                 B                 mode                                      
G402S | masked v3 G406R | masked v3 leave_AF_model_out -0.232 -0.096 -0.076   
                                    only_AF_model      -0.043 -0.589 -0.669   
G402S | vanilla   G402S | masked v3 leave_AF_model_out  2.248  1.794  1.334   
                                    only_AF_model       0.860  2.678  4.517   
                  G406R | vanilla   leave_AF_model_out  0.428  0.363  0.268   
                                    only_AF_model      -0.073  0.188  0.568   
G406R | vanilla   G406R | masked v3 leave_AF_model_out  1.588  1.335  0.991   
                                    only_AF_model       0.891  1.901  3.280   
WT | masked v3    G402S | masked v3 leave_AF_model_out  0.375  0.216  0.321   
                                    only_AF_model       0.394  1.029  0.609   
                  G406R | masked v3 leave_AF_model_out  0.142  0.120  0.245   
                                    only_AF_model       0.352  0.440 -0.060   
WT | vanilla      G402S | vanilla   leave_AF_model_out  0.007 -0.149 -0.077   
                                    only_AF_model      -0.372  0.250 -0.038   
                  G406R | vanilla   leave_AF_model_out  0.435  0.214  0.191   
                                    only_AF_model      -0.445  0.438  0.529   
                  WT | masked v3    leave_AF_model_out  1.880  1.429  0.937   
                                    only_AF_model       0.094  1.899  3.869   

AF_model                                                    4      5  
A                 B                 mode                              
G402S | masked v3 G406R | masked v3 leave_AF_model_out -0.268 -0.300  
                                    only_AF_model       0.101  0.229  
G402S | vanilla   G402S | masked v3 leave_AF_model_out  2.091  2.387  
                                    only_AF_model       1.491  0.307  
                  G406R | vanilla   leave_AF_model_out  0.340  0.240  
                                    only_AF_model       0.279  0.677  
G406R | vanilla   G406R | masked v3 leave_AF_model_out  1.483  1.846  
                                    only_AF_model       1.312 -0.141  
WT | masked v3    G402S | masked v3 leave_AF_model_out  0.567  0.415  
                                    only_AF_model      -0.374  0.234  
                  G406R | masked v3 leave_AF_model_out  0.299  0.115  
                                    only_AF_model      -0.274  0.463  
WT | vanilla      G402S | vanilla   leave_AF_model_out -0.024 -0.102  
                                    only_AF_model      -0.249  0.064  
                  G406R | vanilla   leave_AF_model_out  0.316  0.138  
                                    only_AF_model       0.030  0.742  
                  WT | masked v3    leave_AF_model_out  1.500  1.870  
                                    only_AF_model       1.616  0.138

,cohort,A,B,seed_mean_difference,CI_low,CI_high,q_BH
0,all_recycles,WT | vanilla,WT | masked v3,1.651,1.571,1.730,0.000
3,all_recycles,G402S | vanilla,G402S | masked v3,2.076,1.995,2.158,0.000
6,all_recycles,G406R | vanilla,G406R | masked v3,1.601,1.531,1.674,0.000
9,all_recycles,WT | vanilla,G402S | vanilla,-0.047,-0.126,0.029,0.257
12,all_recycles,WT | vanilla,G406R | vanilla,0.246,0.175,0.314,0.000
15,all_recycles,G402S | vanilla,G406R | vanilla,0.293,0.213,0.370,0.000
18,all_recycles,WT | masked v3,G402S | masked v3,0.378,0.296,0.459,0.000
21,all_recycles,WT | masked v3,G406R | masked v3,0.196,0.119,0.267,0.000
24,all_recycles,G402S | masked v3,G406R | masked v3,-0.182,-0.261,-0.103,0.000
27,repository_QC,WT | vanilla,WT | masked v3,1.523,1.444,1.598,0.000


## Focused interpretation
The following text is generated from the executed result tables so it stays synchronized with reruns.

In [8]:
display(Markdown(mp.focused_interpretation(summary, contrasts, CHANNEL, OUT)))

**G406R focus.** The global signal is small relative to within-ensemble dispersion.

- Vanilla: WT/G402S/G406R clashscore medians are 30.400/30.360/30.905; G406R−WT seed-balanced difference is +0.259 (95% CI +0.190 to +0.325). G406R−G402S is +0.328 (95% CI +0.251 to +0.402). Bad-clash medians are 823/822/837; maximum-overlap medians are 1.605/1.607/1.607 Å.

- Masked v3: WT/G402S/G406R clashscore medians are 32.350/32.610/32.450; G406R−WT seed-balanced difference is +0.184 (95% CI +0.108 to +0.255). G406R−G402S is -0.194 (95% CI -0.269 to -0.125). Bad-clash medians are 876/883/879; maximum-overlap medians are 1.620/1.625/1.623 Å.

The current data support modest protocol-associated global shifts, not a distinctive large G406R global clash phenotype. They cannot locate a clash at R406 or exclude a severe local contact diluted in a global score. The existing short local distances still justify parsing atom-level cluster logs: identify R406-involving atom pairs, contact frequencies and overlap magnitudes, with WT/G402S controls. Keep this global comparison as QC; local attribution remains untested.

**AF-model dependence.** The following ranges are effect magnitudes, not new hypothesis tests:

- WT masked−vanilla: individual AF-model mean differences range from +nan to +nan; leaving out one AF model at a time gives +nan to +nan.

- G402S masked−vanilla: individual AF-model mean differences range from +nan to +nan; leaving out one AF model at a time gives +nan to +nan.

- G406R masked−vanilla: individual AF-model mean differences range from +nan to +nan; leaving out one AF model at a time gives +nan to +nan.

Original masking raises the mean within every AF-model stratum. The stratified tables and final-r10 sensitivities should accompany claims based on pooled means.

## Reproducibility record
All generated outputs are confined to this channel’s `dataExtra/molprobity/{tables,figures}`. Existing figures and analyses are not overwritten. Input hashes, missing-model lists, full descriptive/contrast tables, pairing diagnostics and sensitivity results accompany the figures.

In [9]:
record = {'channel': CHANNEL, 'total_downloaded_csvs': total_csv_files,
          'channel_csvs': len(inventory), 'rows': len(raw), 'python': platform.python_version(),
          'bootstrap_replicates': mp.BOOTSTRAPS, 'random_seed': mp.RANDOM_SEED,
          'packages': {p: importlib.metadata.version(p) for p in ['numpy','pandas','scipy','statsmodels','matplotlib','seaborn']}}
(OUT/'tables/run_metadata.json').write_text(json.dumps(record, indent=2)+chr(10))
print('Completed:', OUT.relative_to(ROOT))


Completed: cav12/dataExtra/molprobity
